In [1]:
!pip install yadisk

In [ ]:
import os
import yadisk
from tqdm.notebook import tqdm



TOKEN = "<YOUR_YANDEX_DISK_TOKEN>"
DESTINATION_ROOT = "/Uploaded_Checkpoints"

paths_to_upload = [
    "/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251107_021028",
    "/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251202_085040/checkpoint-200000",
    "/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251202_085040/live_config.json",
    "/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251202_085040/training_config.json"
]

# Увеличиваем таймаут, так как файлы огромные
y = yadisk.Client(token=TOKEN)

class ProgressReader:
    def __init__(self, filename, pbar):
        self.file = open(filename, 'rb')
        self.pbar = pbar

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()

    def read(self, size=-1):
        data = self.file.read(size)
        # ВАЖНОЕ ИЗМЕНЕНИЕ:
        # Мы обновляем бар не на длину прочитанного, а выставляем его
        # в текущую позицию файла. Это лечит глюк с ретраями.
        current_position = self.file.tell()
        self.pbar.n = current_position
        self.pbar.refresh()
        return data

    def close(self):
        self.file.close()

    def __getattr__(self, attr):
        return getattr(self.file, attr)

def ensure_remote_dir(path):
    parts = path.strip("/").split("/")
    current_path = ""
    for part in parts:
        current_path = f"{current_path}/{part}"
        if not y.exists(current_path):
            try:
                y.mkdir(current_path)
            except yadisk.exceptions.PathExistsError:
                pass

def upload_file(local_path, remote_path):
    file_size = os.path.getsize(local_path)
    ensure_remote_dir(os.path.dirname(remote_path))
    
    # Проверяем, может файл уже залит такого же размера (докачка "вручную")
    if y.exists(remote_path):
        remote_meta = y.get_meta(remote_path)
        if remote_meta.size == file_size:
            print(f"Skipping {local_path} (already exists and size matches)")
            return

    with tqdm(total=file_size, unit="B", unit_scale=True, desc=local_path) as pbar:
        try:
            with ProgressReader(local_path, pbar) as f:
                # timeout=None важно для больших файлов
                y.upload(f, remote_path, overwrite=True, timeout=None)
        except Exception as e:
            print(f"\n❌ Error uploading {local_path}: {e}")

def process_path(local_path, base_remote_path):
    base_name = os.path.basename(local_path)
    
    if "run_20251202_085040" in local_path:
        remote_dest = f"{base_remote_path}/run_20251202_085040/{base_name}"
    else:
        remote_dest = f"{base_remote_path}/{base_name}"

    if os.path.isfile(local_path):
        upload_file(local_path, remote_dest)
    
    elif os.path.isdir(local_path):
        for root, _, files in os.walk(local_path):
            for file in files:
                full_local = os.path.join(root, file)
                rel_path = os.path.relpath(full_local, local_path)
                full_remote = f"{remote_dest}/{rel_path}"
                upload_file(full_local, full_remote)

if not y.check_token():
    print("Invalid Token")
else:
    print("Token OK. Starting upload...")
    ensure_remote_dir(DESTINATION_ROOT)
    for path in paths_to_upload:
        process_path(path, DESTINATION_ROOT)

Token OK. Starting upload...
Skipping /disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251107_021028/live_config.json (already exists and size matches)
Skipping /disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251107_021028/training_config.json (already exists and size matches)
Skipping /disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251107_021028/checkpoint-40000/model.pt (already exists and size matches)


/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251202_085040/checkpoint-200000/model.pt: …

/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251202_085040/live_config.json:   0%|     …

/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251202_085040/training_config.json:   0%| …